# bank-full.csv: Data Boss session

Exported from Data Boss. Run the cells from top to bottom next to the data file to repeat every cleaning step and chart. Each step has the message the app showed when it was applied.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Load the data

Reads `bank-full.csv` exactly as Data Boss did.

In [ ]:
df = pd.read_csv('bank-full.csv', sep=';')
df.shape

## Cleaning steps

### Step 1

Added column 'contacted_before' = CASE WHEN pdays = -1 THEN 'no' ELSE 'yes' END.

Rows × columns: 45,211 × 17 → 45,211 × 18

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['contacted_before'] = con.sql("""SELECT (CASE WHEN pdays = -1 THEN 'no' ELSE 'yes' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 2

Replaced 36954 occurrence(s) of '-1.0' with missing (NaN) in 'pdays'.

Rows × columns: 45,211 × 18 → 45,211 × 18

In [ ]:
df['pdays'] = df['pdays'].replace(-1.0, np.nan)
df.shape

## Result

In [ ]:
df = df.reset_index(drop=True)
df.head(10)

## Charts

Drawn from the cleaned data above.

### How many customers subscribed

- 2 distinct values across 45,211 rows.
- Most common: no (39,922, 88%).

In [ ]:
s = df['y'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(9.0, 5.0))
ax.bar(s.index.astype(str), s.values)
plt.tight_layout()
plt.show()

### Call duration (seconds, log scale) by outcome

- Highest median: yes (426). Lowest: no (164).
- Overall median 180; middle 50% between 103 and 319.
- 3,235 point(s) fall outside the whiskers (1.5 × IQR).

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.0))
order = df.groupby('y')['duration'].median().sort_values(ascending=False).index[:12]
sns.boxplot(data=df[df['y'].isin(order)], x='y', y='duration', order=order, ax=ax)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

### Customers by result of the previous campaign

- Highest: unknown (36,959). Lowest: success (1,511).
- unknown alone accounts for 82% of the total.

In [ ]:
s = df['poutcome'].value_counts().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9.0, 5.0))
ax.bar(s.index.astype(str), s.values)
plt.tight_layout()
plt.show()

## Save the cleaned data

In [ ]:
df.to_csv('bank-full_cleaned.csv', index=False)